In [15]:
from langgraph.graph import StateGraph, START, END 
from pydantic import BaseModel, Field
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [13]:
# create 3 diff llm
load_dotenv()
generator_llm=0
evaluator_llm=0
optimizer_llm=0

In [ ]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved","needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [ ]:
def generate_tweet(state: TweetState):
    prompt=f"write prompt to generate tweet"

    response=generator_llm.invoke(prompt).conent

    return {"tweet": response}


In [ ]:
class TweetEvaluaionSchema(BaseModel):
    evaluation: Literal["approved","needs_improvement"] = Field(description="final evaluation result")
    feedback: str = Field(description="constructive feedback for the tweet")

structured_evaluator_llm=evaluator_llm.with_structured_output(TweetEvaluaionSchema)


def evaluate_tweet(state: TweetState):
    prompt=f"write prompt to evaluate tweet"

    response=structured_evaluator_llm.invoke(prompt)

    return {"evaluation": response.evaluation, "feedback": response.feedback}

In [ ]:
def optimize_tweet(state: TweetState):
    prompt=f"write prompt for twer optimization"

    response=optimizer_llm.invoke(prompt).content
    iteration=state["iteration"]+1

    return {"tweet": response, "iteration": iteration}

In [ ]:
def route_evaluation(state: TweetState):
    if state["evaluation"] == "approved" or state["iteration"]>=state["max_iteration"]:
        return END
    else:
        return "optimize"

In [ ]:
mygraph=StateGraph(TweetState)

mygraph.add_node("generate", generate_tweet)
mygraph.add_node("evaluate", evaluate_tweet)
mygraph.add_node("optimize", optimize_tweet)

mygraph.add_edge(START, "generate")
mygraph.add_edge("generate", "evaluate")

mygraph.add_conditional_edges("evaluate", route_evaluation)
mygraph.add_edge("optimize", "evaluate")

workflow=mygraph.compile()

In [ ]:
initial_state={
    "topic": "indian_railways",
    "iteration": 0,
    "max_iteration": 5
}

workflow.invoke(initial_state)